# 🚀 E2E Pipeline Phase 2 (Local Wrapper)
Notebook này được thiết kế như một **Wrapper** để chạy thử logic của Phase 2 trên máy Local. 

**Nguyên tắc triển khai:**
- **Import tối đa:** Tái sử dụng trực tiếp các hàm nạp tài nguyên, tiền xử lý và lưu trữ từ `spark_jobs/lda_job.py` và `algorithms/count_min_sketch.py`.
- **Minimal Local Overrides:** Chỉ thay thế module `pyspark.ml` bằng `gensim` cho bước huấn luyện mô hình để có thể chạy không cần Spark Cluster.
- **Schema Integrity:** Đảm bảo đầu ra (CSV/JSON) khớp 100% với `data_flow_schema_evolution.md` thông qua việc dùng chung hàm export của project.

In [8]:
import os, sys, json, torch
import pandas as pd
from datetime import datetime, timezone
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. SETUP ĐƯỜNG DẪN ĐỂ IMPORT ĐƯỢC MODULE NỘI BỘ
current_dir = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, "..")) if os.path.basename(current_dir) == "notebooks" else current_dir
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 2. IMPORT TOÀN BỘ LOGIC TỪ PRODUCTION FILES
# Chúng ta lấy các hàm nạp tài nguyên, tiền xử lý và các hằng số cấu hình
from spark_jobs.lda_job import (
    load_stopwords, 
    load_slang_dict, 
    preprocess_vietnamese_text, # Hàm tiền xử lý chuẩn Python dùng cho cả Spark UDF và Local
    STOPWORDS_PATH, 
    SLANG_DICT_PATH,
    DEFAULT_K,
    DEFAULT_MAX_ITER
)
from algorithms.count_min_sketch import CountMinSketch, export_keyword_freq_csv

# Nạp tài nguyên chuẩn bằng chính các hàm sẽ chạy trên server
stopwords = load_stopwords(os.path.join(PROJECT_ROOT, STOPWORDS_PATH))
slang_dict = load_slang_dict(os.path.join(PROJECT_ROOT, SLANG_DICT_PATH))

print(f"\n✅ Đã đồng bộ logic thành công từ project codebase!")
print(f"✅ Stopwords: {len(stopwords):,} | Slang: {len(slang_dict):,}")

2026-05-01 00:36:57,513 [INFO] lda_job: Loaded 1,942 stopwords from c:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\MassiveDatasets\Project BigData\topic-modeling\data/stopwords_vi.txt
2026-05-01 00:36:57,516 [INFO] lda_job: Loaded 2,124 slang entries from c:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\MassiveDatasets\Project BigData\topic-modeling\data/slang_dict.json



✅ Đã đồng bộ logic thành công từ project codebase!
✅ Stopwords: 1,942 | Slang: 2,124


### 📂 Bước 1: Load dữ liệu & Tiền xử lý chuẩn
*Ở đây ta sử dụng trực tiếp hàm `preprocess_vietnamese_text` import từ `lda_job.py` để đảm bảo kết quả tokenization trên Local và Spark là trùng khớp.*

In [9]:
# 3. LOAD DATA & PREPROCESSING
data_path = os.path.join(PROJECT_ROOT, 'data', 'preprocessed', 'stg_posts_core_vnexpress_sample.csv')
print(f"Đang nạp dữ liệu: {data_path}")
df = pd.read_csv(data_path)

# Sử dụng hàm tiền xử lý 'chính chủ' từ lda_job.py
# Vì dữ liệu sample đã được clean một phần, ta chỉ cần tokenize và lọc stopwords
def local_preprocess(text):
    return preprocess_vietnamese_text(text, stopwords, slang_dict)

df['tokens'] = df['segmented_text'].apply(local_preprocess)
df = df[df['tokens'].apply(len) >= 3].reset_index(drop=True)

print(f"-> Dữ liệu sẵn sàng: {len(df):,} văn bản hợp lệ.")
df[['segmented_text', 'tokens']].head(2)

Đang nạp dữ liệu: c:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\MassiveDatasets\Project BigData\topic-modeling\data\preprocessed\stg_posts_core_vnexpress_sample.csv
[SlangNormalizer] WARNING: Slang dict not found: data/slang_dict.json


2026-05-01 00:37:00,253 [INFO] lda_job: Using shared TextPreprocessor from preprocessing/.


[TextPreprocessor] WARNING: Stopwords file not found: data/stopwords_vi.txt
-> Dữ liệu sẵn sàng: 188 văn bản hợp lệ.


,segmented_text,tokens
0,chuyên_gia rò_rỉ onleaks hợp_tác trang công_ng...,"[chuyên_gia, rò_rỉ, onleaks, hợp_tác_trang, cô..."
1,v70 series vivo thực_hiện thay_đổi phân_cấp tr...,"[v70, series, vivo, thực_hiện, thay_đổi, phân_..."


### 🧠 Bước 2: Huấn luyện LDA (Gensim Local Override)
*Vì `pyspark.ml` không chạy được trên local thiếu Spark, ta dùng Gensim làm 'máy ảo' huấn luyện.*

In [10]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel

print("Huấn luyện LDA (Gensim Local)...")
dictionary = Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(text) for text in df['tokens']]

# Tham số K và Passes lấy từ hằng số chuẩn của project
K = DEFAULT_K
passes = 10 # Cho local chạy nhanh

lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=K, passes=passes, random_state=42)

print(f"-> Tìm thấy {K} topics. Top 3 keywords mỗi topic:")
for idx, topic in lda_model.print_topics(num_topics=K, num_words=3):
    print(f"  {topic}")

2026-05-01 00:37:03,542 [INFO] gensim.corpora.dictionary: adding document #0 to Dictionary<0 unique tokens: []>
2026-05-01 00:37:03,572 [INFO] gensim.corpora.dictionary: built Dictionary<4841 unique tokens: ['android', 'ar1', 'camera', 'cao_cấp', 'chip']...> from 188 documents (total 44067 corpus positions)
2026-05-01 00:37:03,576 [INFO] gensim.utils: Dictionary lifecycle event {'msg': "built Dictionary<4841 unique tokens: ['android', 'ar1', 'camera', 'cao_cấp', 'chip']...> from 188 documents (total 44067 corpus positions)", 'datetime': '2026-05-01T00:37:03.576850', 'gensim': '4.4.0', 'python': '3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'event': 'created'}
2026-05-01 00:37:03,598 [INFO] gensim.models.ldamodel: using symmetric alpha at 0.1
2026-05-01 00:37:03,599 [INFO] gensim.models.ldamodel: using symmetric eta at 0.1
2026-05-01 00:37:03,602 [INFO] gensim.models.ldamodel: using serial LDA version on this

Huấn luyện LDA (Gensim Local)...


2026-05-01 00:37:03,769 [INFO] gensim.models.ldamodel: -10.283 per-word bound, 1245.6 perplexity estimate based on a held-out corpus of 188 documents with 44067 words
2026-05-01 00:37:03,769 [INFO] gensim.models.ldamodel: PROGRESS: pass 0, at document #188/188
2026-05-01 00:37:03,888 [INFO] gensim.models.ldamodel: topic #1 (0.100): 0.012*"máy" + 0.010*"sử_dụng" + 0.008*"có_thể" + 0.008*"sản_phẩm" + 0.007*"hỗ_trợ" + 0.007*"đồng" + 0.007*"màn_hình_hình" + 0.007*"khả_năng" + 0.007*"điện_thoại" + 0.006*"giúp"
2026-05-01 00:37:03,889 [INFO] gensim.models.ldamodel: topic #2 (0.100): 0.014*"iphone" + 0.013*"máy" + 0.011*"có_thể" + 0.010*"sử_dụng" + 0.008*"chuyên_nghiệp" + 0.008*"apple" + 0.007*"hỗ_trợ" + 0.006*"màn_hình_hình" + 0.006*"thiết_bị" + 0.006*"sản_phẩm"
2026-05-01 00:37:03,892 [INFO] gensim.models.ldamodel: topic #9 (0.100): 0.014*"iphone" + 0.012*"chuyên_nghiệp" + 0.010*"có_thể" + 0.010*"apple" + 0.009*"máy" + 0.008*"điện_thoại" + 0.008*"giá" + 0.008*"sử_dụng" + 0.008*"dòng" + 0.00

-> Tìm thấy 10 topics. Top 3 keywords mỗi topic:
  0.016*"find" + 0.016*"màn_hình_hình" + 0.015*"có_thể"
  0.014*"máy" + 0.011*"sản_phẩm" + 0.010*"triệu"
  0.017*"lau" + 0.014*"hút" + 0.013*"sử_dụng"
  0.012*"giúp" + 0.012*"thiết_bị" + 0.011*"hệ_thống"
  0.020*"máy" + 0.019*"chuyên_nghiệp" + 0.012*"hỗ_trợ"
  0.040*"iphone" + 0.018*"apple" + 0.018*"chuyên_nghiệp"
  0.045*"ảnh" + 0.039*"chụp" + 0.019*"ultra"
  0.009*"máy" + 0.007*"core" + 0.007*"ultra"
  0.025*"máy" + 0.013*"neo" + 0.011*"air"
  0.025*"iphone" + 0.017*"apple" + 0.016*"điện_thoại"


### 📊 Bước 3 & 4: Đánh giá & Count-Min Sketch (Task 2.2 - 2.4)

In [11]:
from gensim.models import CoherenceModel

# Task 2.2: Coherence
coherence_model = CoherenceModel(model=lda_model, texts=df['tokens'], dictionary=dictionary, coherence='c_v')
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score (C_v): {coherence_score:.4f}")

# Task 2.3 & 2.4: CMS & Evaluation
cms = CountMinSketch(d=5, w=2048)
exact_counts = {}
for tokens in df['tokens']:
    for word in tokens:
        exact_counts[word] = exact_counts.get(word, 0) + 1
        cms.add(word)

print(f"CMS Total Count: {cms.total_count:,} | Observed Epsilon: {(sum([abs(cms.query(w)-exact_counts[w]) for w in exact_counts])/cms.total_count):.6f}")

2026-05-01 00:37:05,889 [INFO] gensim.topic_coherence.probability_estimation: using ParallelWordOccurrenceAccumulator<processes=7, batch_size=64> to estimate probabilities from sliding windows
2026-05-01 00:37:20,239 [INFO] gensim.topic_coherence.text_analysis: 1 batches submitted to accumulate stats from 64 documents (17200 virtual)
2026-05-01 00:37:20,247 [INFO] gensim.topic_coherence.text_analysis: 2 batches submitted to accumulate stats from 128 documents (29118 virtual)
2026-05-01 00:37:21,352 [INFO] gensim.topic_coherence.text_analysis: 7 accumulators retrieved from output queue
2026-05-01 00:37:21,362 [INFO] gensim.topic_coherence.text_analysis: accumulated word occurrence stats for 32779 virtual documents


Coherence Score (C_v): 0.4483
CMS Total Count: 44,067 | Observed Epsilon: 0.183970


### 💾 Bước 5: Lưu trữ đầu ra (Sử dụng hàm export chuẩn)
*Ở đây ta sử dụng `export_keyword_freq_csv` import từ `algorithms.count_min_sketch` để ghi dữ liệu chuẩn 100% Data Flow.*

In [12]:
output_dir = os.path.join(PROJECT_ROOT, 'output')
os.makedirs(output_dir, exist_ok=True)
now_utc = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# 1. Lưu stg_topics (JSON) - Mô phỏng logic lda_job.py
stg_topics = []
for idx, topic_words in lda_model.show_topics(formatted=False, num_topics=K, num_words=10):
    kws = [w for w, p in topic_words]
    stg_topics.append({
        "topic_id": int(idx),
        "label": " | ".join(kws[:3]),
        "top_keywords": kws,
        "coherence_score": float(coherence_score),
        "model_version": "lda_gensim_v1",
        "created_at": now_utc
    })
with open(os.path.join(output_dir, 'topics.json'), 'w', encoding='utf-8') as f:
    json.dump(stg_topics, f, ensure_ascii=False, indent=4)

# 2. Lưu stg_post_topics (CSV) - Mô phỏng logic lda_job.py
assignments = []
for i, bow in enumerate(corpus):
    dist = lda_model.get_document_topics(bow)
    best = max(dist, key=lambda x: x[1]) if dist else (0, 0.0)
    assignments.append({
        "post_id": df.iloc[i].get('post_id', f"doc_{i}"),
        "topic_id": int(best[0]),
        "topic_probability": float(best[1]),
        "model_type": "lda",
        "predicted_at": now_utc
    })
pd.DataFrame(assignments).to_csv(os.path.join(output_dir, 'stg_post_topics.csv'), index=False)

# 3. Lưu stg_keyword_freq (SỬ DỤNG HÀM IMPORT TRỰC TIẾP)
export_keyword_freq_csv(
    cms=cms,
    candidates=list(exact_counts.keys()),
    output_path=os.path.join(output_dir, 'stg_keyword_freq.csv'),
    window_start=(datetime.now() - pd.Timedelta(hours=1)).isoformat(),
    window_end=datetime.now().isoformat(),
    source="vnexpress"
)

print(f"\n🎉 HOÀN TẤT! Dữ liệu đã được lưu chuẩn xác vào thư mục: {output_dir}")


🎉 HOÀN TẤT! Dữ liệu đã được lưu chuẩn xác vào thư mục: c:\Users\Admin\OneDrive - ptit.edu.vn\nam4ky2\MassiveDatasets\Project BigData\topic-modeling\output
